# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a **Signal Analysis** task. We aren't predicting a future yes/no label or grouping items; we are trying to find which signals travel together with high visibility (ranking) and engagement (CTR). We want to output grouped effect sizes, correlations, and feature importance to understand how different signals impact SEO performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Our primary targets for analysis are **`avg_position`** and **`ctr`**. Both are directly observed outcomes from Google Search Console data over the 90-day window, not artificial labels generated from rules. We will investigate how various signals (word count, engagement rate, intent) relate to these observed outcomes.

In [2]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("Target Variables Overview:")
print(df[['avg_position', 'ctr']].describe())

Target Variables Overview:
       avg_position           ctr
count   30000.00000  30000.000000
mean       16.34238      0.510733
std        15.21679      3.279162
min         0.00000      0.000000
25%         6.20000      0.000000
50%        10.80000      0.070000
75%        22.30000      0.290000
max       245.00000    100.000000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

For signal analysis, 'good' means finding strong, robust, and statistically significant relationships. Our success metrics will be **Spearman's rank correlation coefficient** (since relationships might be non-linear) and **Feature Importance** scores from a Random Forest regressor (to capture non-linear and interactive effects between signals). A 'good' signal will have a high relative importance and a consistent correlation across different content types.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Checking Spearman correlation of a few continuous features against our target
continuous_cols = ['word_count', 'search_volume', 'engagement_rate', 'avg_position', 'ctr']
print("Spearman Correlation with targets:")
print(df[continuous_cols].corr(method='spearman')[['avg_position', 'ctr']])

Spearman Correlation with targets:
                 avg_position       ctr
word_count           0.201764  0.069178
search_volume        0.058766 -0.075574
engagement_rate     -0.045114  0.407057
avg_position         1.000000 -0.144447
ctr                 -0.144447  1.000000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = **one pseudonymized content item** (`content_id`) for a specific client (`client_id`) over a 90-day trailing window.

In [4]:
print(f"Unit of analysis check:\n")
print(f"Rows: {len(df)}")
print(f"Unique content IDs: {df['content_id'].nunique()}")
print(f"Unique client IDs: {df['client_id'].nunique()}")
print("\nFirst row showing the grain:")
display(df[['content_id', 'client_id', 'content_type', 'avg_position', 'ctr']].head(1))

Unit of analysis check:

Rows: 30000
Unique content IDs: 30000
Unique client IDs: 32

First row showing the grain:


,content_id,client_id,content_type,avg_position,ctr
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,0.76


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like 'longer content ranks better' fails because signals are highly tangled. Search volume influences click-through rates, intent changes the expected engagement, and a high word count might be irrelevant for transactional pages but crucial for informational articles. An interpretable ML model (like a tree-based model) can isolate these interactions and highlight the true underlying relative importance of each signal, which an if-statement cannot.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Show how 'word_count' correlation changes by 'content_type', proving a simple global rule fails.
for ctype in df['content_type'].dropna().unique():
    subset = df[df['content_type'] == ctype]
    corr = subset['word_count'].corr(subset['avg_position'], method='spearman')
    print(f"Word count vs Position Spearman correlation for '{ctype}': {corr:.3f}")

Word count vs Position Spearman correlation for 'keyword article': 0.167
Word count vs Position Spearman correlation for 'feedly article': -0.042
Word count vs Position Spearman correlation for 'comparison article': 0.214


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.